In [2]:
import psycopg2
import os
import io
from dotenv import load_dotenv
import pandas as pd
import sqlalchemy
import matplotlib.pyplot as plt

In [3]:
load_dotenv(dotenv_path="../../.env")

host=os.getenv("host")
dbname=os.getenv("dbname")
user=os.getenv("user")
password=os.getenv("password")
port=os.getenv("port")

engine = sqlalchemy.create_engine(f'postgresql://{user}:{password}@{host}:{port}/{dbname}')


1) Quais clientes do segmento Premium estão cadastrados e qual é o limite de crédito de cada um?

In [4]:
query_1 = """
        SELECT
        nome,
        segmento,
        limite_credito
        FROM case_3.clientes_silver
        WHERE segmento = 'premium'
        """
Q1 = pd.read_sql(query_1, engine)

Q1

,nome,segmento,limite_credito
0,roberto nunes,premium,50000.0
1,lucas ferreira,premium,75000.0
2,larissa cardoso,premium,60000.0
3,andré moreira,premium,80000.0
4,amanda vieira,premium,55000.0


2) Qual é o valor total faturado por estado em pedidos com status "Entregue"? Ordene do maior para o menor.

In [5]:
query_2 = """
SELECT 
t2.estado,
sum(t1.valor_total - t1.desconto) AS valor_liquido
FROM case_3.pedidos_silver AS t1
LEFT JOIN case_3.clientes_silver AS t2
ON t1.cliente_id = t2.cliente_id
WHERE t1.status = 'entregue'
GROUP BY estado
ORDER BY valor_liquido DESC

"""
Q2 = pd.read_sql(query_2, engine)

Q2

,estado,valor_liquido
0,MS,126390.0
1,DF,99250.0
2,PR,48670.0
3,CE,45075.0
4,PB,39650.0
5,MG,21885.0
6,ES,20290.0
7,PE,12440.0
8,RN,11840.0
9,MT,11685.0


3) Quais clientes fizeram pedidos cujo valor total ultrapassou seu próprio limite de crédito em um único pedido?

In [15]:
query_3 = """
SELECT 
t1.cliente_id,
t2.nome,
max(t1.valor_total) AS valor_max,
t2.limite_credito,
max(t1.valor_total)-t2.limite_credito AS delta
FROM case_3.pedidos_silver AS t1
INNER JOIN case_3.clientes_silver AS t2
ON t1.cliente_id = t2.cliente_id
GROUP BY t2.nome, t1.cliente_id, t2.limite_credito
HAVING max(t1.valor_total)-t2.limite_credito > 0
"""
Q3 = pd.read_sql(query_3, engine)

Q3

,cliente_id,nome,valor_max,limite_credito,delta


4) Para cada vendedor, qual foi o mês com maior volume de vendas em pedidos entregues?

In [36]:
query_4 = """
WITH t1 AS (
SELECT 
vendedor,
data_entrega,
pedido_id,
status	
FROM case_3.pedidos_silver
WHERE data_entrega != '-' 
AND status = 'entregue'),

t2 AS (SELECT 
vendedor,
substr(data_entrega, 1, 7) AS mes,
count(pedido_id) AS qtd_pedidos,
ROW_NUMBER() OVER (PARTITION BY vendedor ORDER BY COUNT(pedido_id) DESC) AS ordem
FROM t1
GROUP BY vendedor, mes
ORDER BY vendedor, mes)

SELECT 
vendedor,
mes,
qtd_pedidos
FROM t2
WHERE ordem = 1

"""
Q4 = pd.read_sql(query_4, engine)

Q4

,vendedor,mes,qtd_pedidos
0,ana paula,2023-10,4
1,bruno,2023-10,3
2,carlos,2023-11,6
3,mariana,2023-06,5


5) Identifique clientes com ao menos 2 pedidos em 2023 cujo último pedido foi há mais de 90 dias da data mais recente na base.

In [82]:
query_5 = """
WITH t1 AS (SELECT 
tab1.cliente_id,
tab2.nome,
count(*) AS qtd_pedidos
FROM case_3.pedidos_silver AS tab1
INNER JOIN case_3.clientes_silver AS tab2
ON tab1.cliente_id = tab2.cliente_id
WHERE TO_CHAR(tab1.data_pedido, 'YYYY') = '2023' 
GROUP BY tab1.cliente_id, tab2.nome
HAVING count(*) >=2),

t2 AS (SELECT tab1.cliente_id,
tab2.nome,
min(EXTRACT(DAY FROM((SELECT max(data_pedido) FROM case_3.pedidos_silver) - tab1.data_pedido))) AS min_delta

FROM case_3.pedidos_silver AS tab1
INNER JOIN case_3.clientes_silver AS tab2
ON tab1.cliente_id = tab2.cliente_id
GROUP BY tab1.cliente_id, tab2.nome
HAVING min(EXTRACT(DAY FROM((SELECT max(data_pedido) FROM case_3.pedidos_silver) - tab1.data_pedido))) > 90)

SELECT t1.* , t2.min_delta
FROM t1
INNER JOIN t2
ON t1.cliente_id = t2.cliente_id

"""
Q5 = pd.read_sql(query_5, engine)

Q5

,cliente_id,nome,qtd_pedidos,min_delta
0,26,tatiana freitas,2,96.0
1,6,fernanda lima,2,147.0
2,13,eduardo gomes,2,141.0
3,1,joão silva,4,117.0
4,29,henrique costa,2,93.0
5,15,thiago carvalho,2,105.0
6,3,pedro alves,2,150.0
7,22,bruna lopes,2,99.0
8,9,marcos oliveira,2,144.0
9,19,gustavo pinto,2,102.0
